In [ ]:
%load_ext autoreload
%autoreload 2

## This notebook divides the GFA dataset into 5°x5° tiles and fits both static and time-varying distributions to them. I've included this notebook for completeness but we ended up moving away from all of this.

In [ ]:
from wildfire_stats import config as cfg
from wildfire_stats.data import io as dio
from wildfire_stats.old import spatial_analysis as sa

P = cfg.load_paths()
A = cfg.load_assumptions()

In [ ]:
gfa_all = dio.load_gfa(P.gfa)

gfa_all.head()

Divide GFA global data into 5x5° grid cells. For each cell that has >400 fires of a given land cover type, run the fitting routine statically.

In [ ]:
results_gfa = sa.analyze_gfa_by_grid(
    gfa_all, xmin=A["min_fire"], R=25, min_n=A["min_fires"], random_state=A["random_seed"]
)

dio.save_pickle(results_gfa, P.output / "gfa_spatial_static_by_biome.pkl")

results_gfa

Take all likelihood tables and parameters and apply our logic to record what the "good" fits are for each tile+biome.

In [ ]:
best_fits_gfa = sa.filter_best_fits(results_gfa)
best_fits_gfa_no_reductions = sa.filter_best_fits_include_reductions(results_gfa)
best_fits_gfa

A variety of visualizations...

In [ ]:
# Note this one uses the version of "good" fits that does not filter out fits that reduce to a simpler one
sa.plot_distribution_fractions_cells(best_fits_gfa_no_reductions)

In [ ]:
sa.plot_biome_bestfit_pies(best_fits_gfa, "Woody savannas")

In [ ]:
sa.plot_gp_vs_tpl(best_fits_gfa, "Woody savannas")

In [ ]:
# TODO(bug): this matches an older 3-argument signature; against the current
# plot_parameter_heatmap(best_fits_gfa, df_tv, biome, distribution) it binds
# df_tv="Savannas" and draws nothing.
sa.plot_parameter_heatmap(best_fits_gfa, "Savannas", "truncated_power_law")

In [ ]:
sa.plot_distribution_params_biome(best_fits_gfa, "Mixed forest", color_by="lon")

Below is the time-varying section where fires of a specified biome in each cell are binned by years (such that at least 75% of the bins have min_per_bin fires) and fit statically, then the parameters are fit with OLS to find their change over time. Many tiles do not have enough fires to be binned, so a time-varying fit is not run on them.

In [ ]:
time_fits_tpl_grid = sa.analyze_time_varying_fits_grid_single_biome(
    gfa_all,
    results_gfa,
    best_fits_gfa,
    target_biome="Grasslands",
    target_distribution="lognormal",
    xmin=A["min_fire"],
    R_boot=5,
    min_per_bin=A["min_fires"],
    max_bin_width=3
)

dio.save_pickle(time_fits_tpl_grid, P.output / "gfa_spatial_time_varying_grasslands_lognormal.pkl")

In [ ]:
df_tv = sa.time_fits_grid_to_df(time_fits_tpl_grid)
df_tv.head()

In [ ]:
sa.plot_parameter_heatmap(
    best_fits_gfa,
    df_tv,
    biome="Open shrublands",
    distribution="lognormal"
)

In [ ]:
example_tile = list(time_fits_tpl_grid.keys())[0]
sa.plot_timevary_param_space_grid(time_fits_tpl_grid, example_tile)